# Correlation analysis flattening

In [ ]:
"""
File Name: correlation_analysis_flattening.ipynb

Author:
    Alessio Morea
    B-Open Solutions srl

Date: August 2025

Description:
------------
This notebook provides a comparative analysis of the Gamma Naught backscatter coefficient.
It evaluates the impact of the Radiometric Terrain Correction (RTC) flattening step on the final
Normalized Radar Backscatter (NRB) product.
The analysis compares datasets from two different acquisition geometries (e.g., ascending and descending orbits).

Usage:
------
To use this script, define the following parameters:

    - s1_path: Path to the NRB product from the first acquisition geometry with terrain flattening applied.
    - s1_nf_path: Path to the NRB product from the first acquisition geometry without terrain flattening applied.
    - s2_path: Path to the NRB product from the second acquisition geometry with terrain flattening applied.
    - s2_nf_path: Path to the NRB product from the second acquisition geometry without terrain flattening applied.
    - pol: The radar polarization channel to analyze (e.g., 'V:V', 'V:H').
    - spacing: The side length, in pixels, of the square patches to be extracted for statistical comparison.
"""

In [ ]:
import rosel.io
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 20})
plt.rcParams["lines.markersize"] = 12
plt.rcParams["font.family"] = "serif"
cmap = plt.get_cmap("tab10")
colors = [cmap(i) for i in range(cmap.N)]
markers = ["s", "^", "v", "s", "P", "*", "X", "d"]
ls = ["--", "-", "dotted", "-."]
ms = 10
alpha = 0.7

In [ ]:
s1_path = "/home/morea/SENTINEL1/SALZBURG/44/ESA/WDIR_DYNAMIC/nrb"
s1_nf_path = "/home/morea/SENTINEL1/SALZBURG/44/ESA/WDIR_DYNAMIC_NO_FLATTENING/nrb"
s2_path = "/home/morea/SENTINEL1/SALZBURG/95/ESA/WDIR_DYNAMIC/nrb"
s2_nf_path = "/home/morea/SENTINEL1/SALZBURG/95/ESA/WDIR_DYNAMIC_NO_FLATTENING/nrb"

In [ ]:
s1_44 = rosel.io.read(s1_path, consolidate_metadata=False)
s1_44_nf = rosel.io.read(s1_nf_path, consolidate_metadata=False)

s1_95 = rosel.io.read(s2_path, consolidate_metadata=False)
s1_95_nf = rosel.io.read(s2_nf_path, consolidate_metadata=False)

In [ ]:
pol = "V:V"
spacing = 1000

In [ ]:
mask = (s1_44.mask == 0).compute()
lia_44 = s1_44.lia.where(mask, np.nan).persist()
mosaic_44 = s1_44.gamma0.sel(polarization=pol).where(mask, np.nan).persist()
mosaic_44_nf = s1_44_nf.gamma0.sel(polarization=pol).where(mask, np.nan).persist()

In [ ]:
mask = ((mosaic_44[0] > 1e-5) * (mosaic_44_nf[0] > 1e-5)).compute()
m_corr = (mosaic_44[0].where(mask, np.nan)).compute()
m_nocorr = (mosaic_44_nf[0].where(mask, np.nan)).compute()

In [ ]:
h1, x, y = np.histogram2d(
    lia_44.values.ravel(),
    m_corr.values.ravel(),
    bins=[np.linspace(0, 1.7, 1000), np.linspace(0, 0.7, 1000)],
    density=True,
)
h1 = h1.T

In [ ]:
h2, x, y = np.histogram2d(
    lia_44.values.ravel(),
    m_nocorr.values.ravel(),
    bins=[np.linspace(0, 1.7, 1000), np.linspace(0, 0.7, 1000)],
    density=True,
)
h2 = h2.T

In [ ]:
n = 100

plt.title("Track 044 after correction")

plt.imshow(h1, vmin=0, vmax=10)

plt.xticks([i for i in range(x.size)[::n]], labels=[f"{i:.1f}" for i in np.rad2deg(x)[::n]], rotation=90)
plt.yticks([i for i in range(y.size)[::n]], labels=[f"{i:.2f}" for i in y[::n]])
plt.xlabel(r"Local Incidence angle [deg]")
plt.ylabel(r"$\gamma^0$")
plt.gca().invert_yaxis()
plt.grid(True, ls="--")

In [ ]:
n = 100
plt.title("Track 044 before correction")
plt.imshow(h2, vmin=0, vmax=10)

plt.xticks([i for i in range(x.size)[::n]], labels=[f"{i:.1f}" for i in np.rad2deg(x)[::n]], rotation=90)
plt.yticks([i for i in range(y.size)[::n]], labels=[f"{i:.2f}" for i in y[::n]])
plt.xlabel(r"Local Incidence angle [deg]")
plt.ylabel(r"$\gamma^0$")
plt.gca().invert_yaxis()
plt.grid(True, ls="--")

In [ ]:
mask = (s1_95.mask == 0).compute()
lia_95 = s1_95.lia.where(mask, np.nan).persist()
mosaic_95 = s1_95.gamma0.sel(polarization=pol).where(mask, np.nan).persist()
mosaic_95_nf = s1_95_nf.gamma0.sel(polarization=pol).where(mask, np.nan).persist()

In [ ]:
def corr(mosaic, lia):
    print("\t", "_" * 50)
    ind = np.where((np.isfinite(mosaic) * mosaic > 1e-5) * np.isfinite(lia))
    if ind[0].size > 0:
        x = mosaic.values[ind].ravel()
        y = lia.values[ind].ravel()
        print(f"\t size: {x.size,y.size}")
        mx = np.nanmean(x)
        my = np.nanmean(y)
        sx = np.nanstd(x)
        sy = np.nanstd(y)
        c = np.nanmean((x - mx) * (y - my)) / (sx * sy)
        print("\t", f"mosaic : {mx} +- {sx}")
        print("\t", f"lia    : {my} +- {sy}")
        print("\t", f"corr   : {c}")
    else:
        print("\t No valid points found")
        c = np.nan
    return c

In [ ]:
result = []
result_nf = []
counter = 0
for i in range(lia_44.y.size // spacing):
    for j in range(lia_44.x.size // spacing):
        print("=" * 50)
        yslice = slice(i * spacing, (i + 1) * spacing)
        xslice = slice(j * spacing, (j + 1) * spacing)
        print(yslice, xslice)
        l = lia_44.isel(y=yslice, x=xslice)
        a = mosaic_44.isel(y=yslice, x=xslice)
        b = mosaic_44_nf.isel(y=yslice, x=xslice)

        f = []
        nf = []
        for d in range(a.acquisition_date.size):
            f.append(corr(a.isel(acquisition_date=d), l))
            nf.append(corr(b.isel(acquisition_date=d), l))

        if all(np.isfinite(f)) * all(np.isfinite(nf)):
            result.append(f)
            result_nf.append(nf)
        counter += 1

for i in range(lia_95.y.size // spacing):
    for j in range(lia_95.x.size // spacing):
        print("=" * 50)
        yslice = slice(i * spacing, (i + 1) * spacing)
        xslice = slice(j * spacing, (j + 1) * spacing)
        print(yslice, xslice)
        l = lia_95.isel(y=yslice, x=xslice)
        a = mosaic_95.isel(y=yslice, x=xslice)
        b = mosaic_95_nf.isel(y=yslice, x=xslice)

        f = []
        nf = []
        for d in range(a.acquisition_date.size):
            f.append(corr(a.isel(acquisition_date=d), l))
            nf.append(corr(b.isel(acquisition_date=d), l))
        if all(np.isfinite(f)) * all(np.isfinite(nf)):
            result.append(f)
            result_nf.append(nf)
result = np.abs(result)
result_nf = np.abs(result_nf)

In [ ]:
print(f"Flattening    : {np.nanmean(result)} +- {np.nanstd(result)}")
print(f"No Flattening : {np.nanmean(result_nf)} +- {np.nanstd(result_nf)}")

In [ ]:
ind = np.where((np.isfinite(result) * np.isfinite(result_nf)))

res = result[ind]
res_nf = result_nf[ind]

In [ ]:
plt.figure(figsize=(15, 7))

plt.plot(res[:], ls="--", marker="o", ms=ms, color="red", alpha=alpha, mec="black", label="Flattening")
plt.plot(res_nf[:], ls="-.", marker="s", ms=ms, color="orange", alpha=alpha, mec="black", label="No Flattening")
plt.grid("True", ls="--")
plt.xlabel("Patch")
plt.ylabel("Correlation index")
plt.legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.15), shadow=True)